# Breaking Secure Communication

---

We built HMAC in the last notebook and it felt secure. HMAC *is* 100% secure against the threats it was designed for. But cryptographic protocols fail in ways that have nothing to do with the strength of the underlying primitive.

We shall now simulate attacker behaviour(for if you wish to defeat an enemy, you must know them first). We will:
- **Record and replay** a valid authenticated packet
- **Compromise a node** and trace the blast radius
- **Intercept a Diffie-Hellman exchange** and impersonate both parties

Each attack ends with a question: *what is the minimal fix?* That question is what motivates every protocol feature.

> **Run every cell in order.** Later cells depend on classes defined earlier.

In [ ]:
# Imports used throughout
import os, hashlib, hmac, secrets, time, random
from collections import defaultdict
print('Ready.' )

Ready.


---
<a id='sec0'></a>
## Setup — The Baseline Secure Channel

This is the channel from Notebook 1, reproduced here so everything is self-contained.
Alice and Bob share a secret key and exchange HMAC-authenticated packets.

```
packet = { message, HMAC(key, message) }
```

Ponder upon this question:

> *What security failures can still occur in an application that correctly implements an unforgeable HMAC?*

In [ ]:
# Baseline authenticated channel (from Notebook 1)

def make_tag(key: bytes, data: bytes) -> bytes:
    return hmac.new(key, data, hashlib.sha256).digest()

def verify_tag(key: bytes, data: bytes, tag: bytes) -> bool:
    return hmac.compare_digest(make_tag(key, data), tag)


class BaseChannel:
    """Simple HMAC-authenticated channel with no replay protection."""

    def __init__(self, name: str, key: bytes):
        self.name = name
        self.key  = key

    def send(self, message: str) -> dict:
        msg_bytes = message.encode()
        tag       = make_tag(self.key, msg_bytes)
        pkt       = {'message': message, 'tag': tag.hex()}
        print(f'[{self.name}] SEND  "{message}"  tag={tag.hex()[:16]}...')
        return pkt

    def receive(self, pkt: dict) -> str | None:
        msg_bytes = pkt['message'].encode()
        tag       = bytes.fromhex(pkt['tag'])
        if verify_tag(self.key, msg_bytes, tag):
            print(f'[{self.name}] RECV  "{pkt["message"]}"  -> ACCEPTED')
            return pkt['message']
        print(f'[{self.name}] RECV  "{pkt["message"]}"  -> REJECTED (bad tag)')
        return None  
# Example
SHARED_KEY = secrets.token_bytes(32)
alice = BaseChannel('Alice', SHARED_KEY)
bob   = BaseChannel('Bob',   SHARED_KEY)

pkt = alice.send('Transfer $100 to Charlie')
bob.receive(pkt)

[Alice] SEND  "Transfer $100 to Charlie"  tag=ce69af039bf2558c...
[Bob] RECV  "Transfer $100 to Charlie"  -> ACCEPTED


'Transfer $100 to Charlie'

---
<a id='sec1'></a>
## 1) Replay Attack

### The Setup

Mallory sits on the network between Alice and Bob.  
She **cannot** forge a valid HMAC (as she does not have the key).  
However, she can **record** every packet completely and **retransmit** it later.

```
Time 0:  Alice  --[Transfer $100 to Charlie, TAG]-->  Bob   (legitimate)
Time 1:  Mallory --[Transfer $100 to Charlie, TAG]-->  Bob   (replay)
Time 2:  Mallory --[Transfer $100 to Charlie, TAG]-->  Bob   (replay again)
```

### Question

> *Bob verifies the HMAC and it passes every time as the HMAC is supposed to be unforgeable. Then what does Bob need in order to detect the replays?*

In [ ]:
# Attack 1a: Mallory replays a captured packet
class Mallory:
    """Passive attacker: records packets and replays them."""

    def __init__(self):
        self.captured = []

    def intercept(self, pkt: dict) -> dict:
        """Forward the packet but keep a copy.
        
        Because pkt is a mutable dict; appending pkt stores a reference to the same object.
        If the original pkt is later modified, the stored item in self.captured will reflect those changes.
        
        pkt.copy() makes a shallow copy (a new dict with the same top-level keys/values), 
        so you store a snapshot of the packet at that moment — safe for replaying later without accidental mutation.
        """
        self.captured.append(pkt.copy())
        print(f'[Mallory] captured "{pkt["message"]}"  tag={pkt["tag"][:16]}...')
        return pkt

    def replay(self, index: int = 0) -> dict:
        """Re-send a previously captured packet unchanged."""
        pkt = self.captured[index]
        print(f'[Mallory] REPLAYING "{pkt["message"]}"')
        return pkt


mallory = Mallory()

print('=== Legitimate exchange ===')
pkt = alice.send('Transfer $100 to Charlie')
pkt = mallory.intercept(pkt)
bob.receive(pkt)

print()
print('=== Mallory replays the same packet twice ===')
bob.receive(mallory.replay(0))
bob.receive(mallory.replay(0))

print()
print('HMAC verified correctly each time; hence Bob cannot tell real from replay.')

=== Legitimate exchange ===
[Alice] SEND  "Transfer $100 to Charlie"  tag=ce69af039bf2558c...
[Mallory] captured "Transfer $100 to Charlie"  tag=ce69af039bf2558c...
[Bob] RECV  "Transfer $100 to Charlie"  -> ACCEPTED

=== Mallory replays the same packet twice ===
[Mallory] REPLAYING "Transfer $100 to Charlie"
[Bob] RECV  "Transfer $100 to Charlie"  -> ACCEPTED
[Mallory] REPLAYING "Transfer $100 to Charlie"
[Bob] RECV  "Transfer $100 to Charlie"  -> ACCEPTED

HMAC verified correctly each time; hence Bob cannot tell real from replay.


### What Just Happened?

The HMAC is **perfectly valid** on every replayed packet.
Unforgeability only guarantees that Mallory cannot *create* new valid `(message, tag)` pairs.
It does not guarantee **freshness** , i.e, whether the message is new or not.

HMAC is only a proof of *origin*, not a proof of *recency*.

---
### Fix 1a — Sequence Counters

Include a monotonically increasing counter in every packet.  
Bob can now reject any packet whose counter is not strictly greater than the last one he accepted.

```
packet = { counter, message, HMAC(key, counter || message) }
```

In [17]:
# Fix 1a: Counter-protected channel

class CounterChannel:
    """HMAC channel with monotonic send counter and replay detection."""

    def __init__(self, name: str, key: bytes):
        self.name    = name
        self.key     = key
        self.send_ctr = 0
        self.recv_ctr = -1   # last accepted counter

    def send(self, message: str) -> dict:
        ctr       = self.send_ctr
        self.send_ctr += 1
        payload   = f'{ctr}:{message}'.encode()
        tag       = make_tag(self.key, payload)
        pkt       = {'counter': ctr, 'message': message, 'tag': tag.hex()}
        print(f'[{self.name}] SEND  ctr={ctr}  "{message}"  tag={tag.hex()[:16]}...')
        return pkt

    def receive(self, pkt: dict) -> str | None:
        ctr       = pkt['counter']
        payload   = f'{ctr}:{pkt["message"]}'.encode()
        tag       = bytes.fromhex(pkt['tag'])

        if not verify_tag(self.key, payload, tag):
            print(f'[{self.name}] RECV  ctr={ctr}  -> REJECTED (bad tag)')
            return None
        if ctr <= self.recv_ctr:
            print(f'[{self.name}] RECV  ctr={ctr}  -> REJECTED (replay! last={self.recv_ctr})')
            return None

        self.recv_ctr = ctr
        print(f'[{self.name}] RECV  ctr={ctr}  "{pkt["message"]}"  -> ACCEPTED')
        return pkt['message']


alice2 = CounterChannel('Alice', SHARED_KEY)
bob2   = CounterChannel('Bob',   SHARED_KEY)

print('=== Legitimate exchange ===')
p0 = alice2.send('Transfer $100 to Charlie')
bob2.receive(p0)
p1 = alice2.send('Transfer $50 to Diana')
bob2.receive(p1)

print()
print('=== Mallory replays packet 0 ===')
bob2.receive(p0)   # counter 0 already seen so rejected

=== Legitimate exchange ===
[Alice] SEND  ctr=0  "Transfer $100 to Charlie"  tag=b68b132344607315...
[Bob] RECV  ctr=0  "Transfer $100 to Charlie"  -> ACCEPTED
[Alice] SEND  ctr=1  "Transfer $50 to Diana"  tag=7a3083a2738ebbf9...
[Bob] RECV  ctr=1  "Transfer $50 to Diana"  -> ACCEPTED

=== Mallory replays packet 0 ===
[Bob] RECV  ctr=0  -> REJECTED (replay! last=1)


### Fix 1b — Nonces

The problem with counters is that they require **state synchronisation** between sender and receiver. If either side reboots or reconnects, the counter must be re-established.

A **nonce** (number used once) is a random value included in each packet.  
Bob keeps a set of all nonces he has seen and rejects any repeat.

```
packet = { nonce (128-bit random), message, HMAC(key, nonce || message) }
```

Differences in counter and nonce:-

| | Counter | Nonce |
|---|---|---|
| State required | Yes | Yes |
| Replay detection | Strict ordering | Set membership |
| Handles reorder? | No | Yes |
| State size | O(1) | O(n) |

In [18]:
# Fix 1b: Nonce-protected channel

class NonceChannel:
    """HMAC channel with random nonces and a seen-nonce cache."""

    def __init__(self, name: str, key: bytes):
        self.name       = name
        self.key        = key
        self.seen_nonces = set()

    def send(self, message: str) -> dict:
        nonce   = secrets.token_bytes(16)   # 128-bit random nonce
        payload = nonce + message.encode()
        tag     = make_tag(self.key, payload)
        pkt     = {'nonce': nonce.hex(), 'message': message, 'tag': tag.hex()}
        print(f'[{self.name}] SEND  nonce={nonce.hex()[:12]}...  "{message}"')
        return pkt

    def receive(self, pkt: dict) -> str | None:
        nonce   = bytes.fromhex(pkt['nonce'])
        payload = nonce + pkt['message'].encode()
        tag     = bytes.fromhex(pkt['tag'])

        if not verify_tag(self.key, payload, tag):
            print(f'[{self.name}] RECV  -> REJECTED (bad tag)')
            return None
        if nonce in self.seen_nonces:
            print(f'[{self.name}] RECV  -> REJECTED (replay! nonce already seen)')
            return None

        self.seen_nonces.add(nonce)
        print(f'[{self.name}] RECV  "{pkt["message"]}"  -> ACCEPTED')
        return pkt['message']


alice3 = NonceChannel('Alice', SHARED_KEY)
bob3   = NonceChannel('Bob',   SHARED_KEY)

print('=== Legitimate exchange ===')
p0 = alice3.send('Transfer $100 to Charlie')
bob3.receive(p0)
p1 = alice3.send('Transfer $50 to Diana')
bob3.receive(p1)

print()
print('=== Mallory replays packet 0 ===')
bob3.receive(p0)

=== Legitimate exchange ===
[Alice] SEND  nonce=b48fbbf95c7a...  "Transfer $100 to Charlie"
[Bob] RECV  "Transfer $100 to Charlie"  -> ACCEPTED
[Alice] SEND  nonce=395798c36595...  "Transfer $50 to Diana"
[Bob] RECV  "Transfer $50 to Diana"  -> ACCEPTED

=== Mallory replays packet 0 ===
[Bob] RECV  -> REJECTED (replay! nonce already seen)


### Fix 1c — Replay Window

As you might have already thought, storing *every* nonce ever seen seems to be expensive for long-lived connections. Moreover, in our project, we have even more severe memory constraints than ordinary scenarios.
As a compromise, real protocols (like IPsec, WireGuard) use a **sliding window**:

- Accept any packet whose sequence number falls within `[highest - W, highest]`
- Track a W-bit bitmask of which slots in the window have been received
- Reject duplicates (already set in bitmask) and out-of-window packets

```
Window W = 64 packets

         oldest                           newest
           |                               |
  ... [seen][seen][    ][seen][    ][seen][N]
              ^--- reject duplicates         ^--- advance window on new high
```

In [19]:
# Fix 1c: Sliding-window replay protection

class WindowChannel:
    """Replay protection with a fixed-size sliding window (like IPsec ESP)."""

    WINDOW = 64

    def __init__(self, name: str, key: bytes):
        self.name     = name
        self.key      = key
        self.send_seq = 0
        self.high     = -1
        self.bitmap   = 0

    def send(self, message: str) -> dict:
        seq     = self.send_seq
        self.send_seq += 1
        payload = seq.to_bytes(8, 'big') + message.encode()
        tag     = make_tag(self.key, payload)
        pkt     = {'seq': seq, 'message': message, 'tag': tag.hex()}
        print(f'[{self.name}] SEND  seq={seq}  "{message}"')
        return pkt

    def receive(self, pkt: dict) -> str | None:
        seq = pkt['seq']

        # Reject malformed sequence numbers
        if not isinstance(seq, int) or seq < 0 or seq >= (1 << 64):
            print(f'[{self.name}] seq={seq}  -> REJECTED (invalid sequence number)')
            return None

        payload = seq.to_bytes(8, 'big') + pkt['message'].encode()
        tag     = bytes.fromhex(pkt['tag'])

        if not verify_tag(self.key, payload, tag):
            print(f'[{self.name}] seq={seq}  -> REJECTED (bad tag)')
            return None

        if seq > self.high:
            shift = seq - self.high
            self.bitmap = (self.bitmap << shift) & ((1 << self.WINDOW) - 1)
            self.high   = seq

        offset = self.high - seq
        if offset >= self.WINDOW:
            print(f'[{self.name}] seq={seq}  -> REJECTED (too old, window={self.WINDOW})')
            return None
        if self.bitmap & (1 << offset):
            print(f'[{self.name}] seq={seq}  -> REJECTED (duplicate)')
            return None

        self.bitmap |= (1 << offset)
        print(f'[{self.name}] seq={seq}  "{pkt["message"]}"  -> ACCEPTED')
        return pkt['message']


alice4 = WindowChannel('Alice', SHARED_KEY)
bob4   = WindowChannel('Bob',   SHARED_KEY)

print('=== Normal delivery ===')
packets = [alice4.send(f'Message {i}') for i in range(5)]
for p in packets:
    bob4.receive(p)

print()
print('=== Replay of packet 2 ===')
bob4.receive(packets[2])

print()
print('=== Out-of-window (too old) ===')
# 1. Capture a legitimately signed packet at seq=5
old_pkt = alice4.send('Ancient message')
# 2. Advance the window by 70 packets so seq=5 falls outside it
for _ in range(70):
    bob4.receive(alice4.send('filler'))
# 3. Deliver the old (genuine) packet -- rejected because 75-5=70 >= WINDOW=64
bob4.receive(old_pkt)

print()
print('=== Malformed sequence number (negative) ===')
bob4.receive({'seq': -1, 'message': 'bad packet', 'tag': 'aa' * 32})

=== Normal delivery ===
[Alice] SEND  seq=0  "Message 0"
[Alice] SEND  seq=1  "Message 1"
[Alice] SEND  seq=2  "Message 2"
[Alice] SEND  seq=3  "Message 3"
[Alice] SEND  seq=4  "Message 4"
[Bob] seq=0  "Message 0"  -> ACCEPTED
[Bob] seq=1  "Message 1"  -> ACCEPTED
[Bob] seq=2  "Message 2"  -> ACCEPTED
[Bob] seq=3  "Message 3"  -> ACCEPTED
[Bob] seq=4  "Message 4"  -> ACCEPTED

=== Replay of packet 2 ===
[Bob] seq=2  -> REJECTED (duplicate)

=== Out-of-window (too old) ===
[Alice] SEND  seq=5  "Ancient message"
[Alice] SEND  seq=6  "filler"
[Bob] seq=6  "filler"  -> ACCEPTED
[Alice] SEND  seq=7  "filler"
[Bob] seq=7  "filler"  -> ACCEPTED
[Alice] SEND  seq=8  "filler"
[Bob] seq=8  "filler"  -> ACCEPTED
[Alice] SEND  seq=9  "filler"
[Bob] seq=9  "filler"  -> ACCEPTED
[Alice] SEND  seq=10  "filler"
[Bob] seq=10  "filler"  -> ACCEPTED
[Alice] SEND  seq=11  "filler"
[Bob] seq=11  "filler"  -> ACCEPTED
[Alice] SEND  seq=12  "filler"
[Bob] seq=12  "filler"  -> ACCEPTED
[Alice] SEND  seq=13  "

---
<a id='sec2'></a>
## Key Compromise

### The Setup

A network of four nodes share keys so any pair can communicate securely.
The naive approach would be to set up one **global shared key** for all nodes.

```
        [A]
       / | \
      /  |  \
    [B]-[C]-[D]

All edges use the same key K.
```

### Question

> *Node A is physically captured. The attacker extracts its key from its memory. Which past and future communications can be compromised?*


In [ ]:
# Attack 2a: Single shared key

class NetworkNode:
    def __init__(self, name: str, keys: dict):
        """
        keys: dict mapping peer_name -> shared_key.
        In the single-key model all peers share the same key.
        """
        self.name = name
        self.keys = keys
        self.log  = []   # (peer, message)

    def send(self, peer: str, message: str) -> dict:
        key     = self.keys[peer]
        payload = message.encode()
        tag     = make_tag(key, payload)
        return {'from': self.name, 'to': peer, 'message': message, 'tag': tag.hex()}

    def receive(self, pkt: dict) -> str | None:
        peer = pkt['from']
        key  = self.keys.get(peer)
        if key is None:
            return None
        tag = bytes.fromhex(pkt['tag'])
        if verify_tag(key, pkt['message'].encode(), tag):
            self.log.append((peer, pkt['message']))
            return pkt['message']
        return None


# Single global key
GLOBAL_KEY = secrets.token_bytes(32)
keys_single = {n: GLOBAL_KEY for n in ['A','B','C','D']}

A = NetworkNode('A', {k:v for k,v in keys_single.items() if k!='A'})
B = NetworkNode('B', {k:v for k,v in keys_single.items() if k!='B'})
C = NetworkNode('C', {k:v for k,v in keys_single.items() if k!='C'})
D = NetworkNode('D', {k:v for k,v in keys_single.items() if k!='D'})
nodes = {'A':A,'B':B,'C':C,'D':D}

# Simulate some traffic (assume this happened in the past)
traffic = [
    A.send('B', 'Sensor reading 42'),
    B.send('C', 'Relay: sensor=42'),
    C.send('D', 'Aggregate OK'),
    D.send('A', 'Ack'),
]
print('=== Past traffic (recorded by attacker on the wire) ===')
for pkt in traffic:
    print(f'  {pkt["from"]} -> {pkt["to"]}: "{pkt["message"]}"')

print()
print('=== Node A is compromised. Attacker has GLOBAL_KEY. ===')
print()
print('Can attacker verify past B->C traffic?')
bc_pkt = traffic[1]
ok = verify_tag(GLOBAL_KEY, bc_pkt['message'].encode(), bytes.fromhex(bc_pkt['tag']))
print(f'  verify B->C: {ok}  <- YES, global key decrypts everything')
print()
print('Can attacker forge a future C->D message?')
forged = {'from':'C','to':'D','message':'Override: shutdown',
          'tag': make_tag(GLOBAL_KEY, b'Override: shutdown').hex()}
result = D.receive(forged)
print(f'  D accepted: "{result}"  <- attack succeeded')

=== Past traffic (recorded by attacker on the wire) ===
  A -> B: "Sensor reading 42"
  B -> C: "Relay: sensor=42"
  C -> D: "Aggregate OK"
  D -> A: "Ack"

=== Node A is compromised. Attacker has GLOBAL_KEY. ===

Can attacker verify past B->C traffic?
  verify B->C: True  <- YES, global key decrypts everything

Can attacker forge a future C->D message?
  D accepted: "Override: shutdown"  <- attack succeeded


### Blast Radius

| Scenario | Single global key | Pairwise keys |
|---|---|---|
| Node A compromised | **All** past + future traffic exposed | Only A's direct links |
| Node A + B compromised | Same | Only A-B traffic |
| Forge B→C message | Yes (have the key) | No (don't have K_BC) |
| Past traffic decryptable | Yes | Yes (if recorded) |

The key insight is that **compromise should be contained**.  
We must ensure that compromising one node does not compromise the whole network.

---
### Fix — Pairwise Keys

Give each pair of nodes its own unique key `K_AB`, `K_AC`, `K_BC`, etc.
For N nodes, this requires `N(N-1)/2` keys.

```
        [A]
      K_AB| \K_AC
          |  \
    [B]--[C]--[D]
      K_BC  K_CD
```

In [21]:
# Fix 2: Pairwise keys

def pairwise_key(name_a: str, name_b: str) -> bytes:
    """Deterministic unique key per pair (in reality: distributed securely)."""
    pair = tuple(sorted([name_a, name_b]))
    return hashlib.sha256(f'K_{pair[0]}_{pair[1]}'.encode()).digest()


node_names = ['A', 'B', 'C', 'D']

def make_pairwise_node(name):
    keys = {peer: pairwise_key(name, peer) for peer in node_names if peer != name}
    return NetworkNode(name, keys)

pA, pB, pC, pD = [make_pairwise_node(n) for n in node_names]
pnodes = {'A': pA, 'B': pB, 'C': pC, 'D': pD}

# Simulate past traffic
past = [
    pA.send('B', 'Sensor reading 42'),
    pB.send('C', 'Relay: sensor=42'),
    pC.send('D', 'Aggregate OK'),
]

print('=== Node A compromised. Attacker has K_AB and K_AC. ===')
print()

compromised_keys = {
    peer: pairwise_key('A', peer) for peer in node_names if peer != 'A'
}

print('Can attacker read A->B past traffic?')
ab_pkt = past[0]
ok = verify_tag(compromised_keys['B'], ab_pkt['message'].encode(), bytes.fromhex(ab_pkt['tag']))
print(f'  YES (K_AB is compromised): {ok}')

print()
print('Can attacker read B->C past traffic (does NOT involve A)?')
bc_pkt = past[1]
# Attacker only has K_AB and K_AC -- not K_BC
for k_name, k_val in compromised_keys.items():
    ok = verify_tag(k_val, bc_pkt['message'].encode(), bytes.fromhex(bc_pkt['tag']))
    if ok:
        print(f'  Verified with K_A{k_name} -- this should not happen!')
print(f'  NO -- B->C uses K_BC which A never had. Attacker is blind.')

print()
print('Blast radius: only edges touching A are exposed. B<->C, B<->D, C<->D are safe.')

=== Node A compromised. Attacker has K_AB and K_AC. ===

Can attacker read A->B past traffic?
  YES (K_AB is compromised): True

Can attacker read B->C past traffic (does NOT involve A)?
  NO -- B->C uses K_BC which A never had. Attacker is blind.

Blast radius: only edges touching A are exposed. B<->C, B<->D, C<->D are safe.


### Scaling Problems

Pairwise keys solve containment but create a **key distribution problem**:

| Nodes (N) | Pairwise keys needed N(N-1)/2 |
|---|---|
| 10 | 45 |
| 100 | 4,950 |
| 1,000 | 499,500 |
| 10,000 | 49,995,000 |

For large wireless sensor networks, this becomes impractical.

Solutions to this will eventually form a part of our overall solution for the key distribution question.

---
<a id='sec3'></a>
## Attack 3 · Man-in-the-Middle on Diffie-Hellman

### Background — Diffie-Hellman Key Exchange

DH lets Alice and Bob agree on a shared secret over an **untrusted channel** with no pre-shared key.

```
Public parameters: prime p, generator g

Alice             chooses secret a         Bob chooses secret b
  |                                          |
  |-- A = g^a mod p  -------------------->  |
  |  <--------------------  B = g^b mod p --|
  |                                          |
  | K = B^a mod p                K = A^b mod p
  |    = g^(ab) mod p           = g^(ab) mod p
```

### The Flaw

> *DH provides confidentiality against a passive eavesdropper. But what can an active attacker do?*

Mallory sits between Alice and Bob and **intercepts both sides of the handshake**.
She runs DH *with each of them separately*, establishing:
- A shared secret `K_AM` with Alice
- A shared secret `K_MB` with Bob

Alice and Bob still think they are talking to each other.

In [ ]:
# DH parameters - small values for illustration ONLY
# (As you have already read,real DH uses 2048-4096 bit primes or elliptic curves)

P = 23   # small prime
G = 5    # generator

def dh_public(private: int, g: int = G, p: int = P) -> int:
    return pow(g, private, p)

def dh_shared(their_public: int, my_private: int, p: int = P) -> int:
    return pow(their_public, my_private, p)

def dh_to_key(shared_int: int) -> bytes:
    """Derive a proper key from the raw DH value."""
    return hashlib.sha256(shared_int.to_bytes(32, 'big')).digest()


print(f'DH parameters: p={P}, g={G}')
print()
print('=== Honest DH (no attacker) ===')
a = secrets.randbelow(P - 2) + 2
b = secrets.randbelow(P - 2) + 2 
A = dh_public(a)
B = dh_public(b)
K_alice = dh_shared(B, a)
K_bob   = dh_shared(A, b)
print(f'Alice secret a={a}, public A=g^a={A}')
print(f'Bob   secret b={b}, public B=g^b={B}')
print(f'Alice computes K = B^a mod p = {K_alice}')
print(f'Bob   computes K = A^b mod p = {K_bob}')
print(f'Keys match: {K_alice == K_bob}')

DH parameters: p=23, g=5

=== Honest DH (no attacker) ===
Alice secret a=12, public A=g^a=18
Bob   secret b=22, public B=g^b=1
Alice computes K = B^a mod p = 1
Bob   computes K = A^b mod p = 1
Keys match: True


In [23]:
# Attack 3: Mallory performs a full MitM on DH

class DHParty:
    """Participant in a DH exchange."""

    def __init__(self, name: str):
        self.name    = name
        self.private = secrets.randbelow(P - 2) + 2
        self.public  = dh_public(self.private)
        self.session_key = None
        print(f'[{name}] private={self.private}, public={self.public}')

    def finish_handshake(self, their_public: int, claimed_from: str):
        raw = dh_shared(their_public, self.private)
        self.session_key = dh_to_key(raw)
        print(f'[{self.name}] received public={their_public} (from "{claimed_from}")')
        print(f'[{self.name}] session key = {self.session_key.hex()[:24]}...')

    def send(self, message: str) -> dict:
        tag = make_tag(self.session_key, message.encode())
        return {'from': self.name, 'message': message, 'tag': tag.hex()}

    def receive(self, pkt: dict) -> str | None:
        tag = bytes.fromhex(pkt['tag'])
        if verify_tag(self.session_key, pkt['message'].encode(), tag):
            print(f'[{self.name}] ACCEPTED "{pkt["message"]}" (believes from {pkt["from"]})')
            return pkt['message']
        print(f'[{self.name}] REJECTED "{pkt["message"]}" -- bad tag')
        return None


class MitMallory:
    """Active MitM: intercepts DH, establishes separate sessions with each party."""

    def __init__(self):
        # Mallory's two private keys - one for each impersonation
        self.priv_to_alice = secrets.randbelow(P - 2) + 2
        self.priv_to_bob   = secrets.randbelow(P - 2) + 2
        self.pub_to_alice  = dh_public(self.priv_to_alice)  # sent to Alice pretending to be Bob
        self.pub_to_bob    = dh_public(self.priv_to_bob)    # sent to Bob pretending to be Alice
        self.key_alice     = None   # shared key with Alice
        self.key_bob       = None   # shared key with Bob
        print(f'[Mallory] priv_a={self.priv_to_alice} pub_a={self.pub_to_alice}')
        print(f'[Mallory] priv_b={self.priv_to_bob}  pub_b={self.pub_to_bob}')

    def intercept_alice_public(self, alice_public: int):
        """Mallory sees Alice's DH public value, computes K_MA."""
        raw = dh_shared(alice_public, self.priv_to_alice)
        self.key_alice = dh_to_key(raw)
        print(f'[Mallory] intercepted Alice public={alice_public} -> K_MA={self.key_alice.hex()[:20]}...')
        # Returns Mallory's public to be forwarded to Bob (as if from Alice)
        return self.pub_to_bob

    def intercept_bob_public(self, bob_public: int):
        """Mallory sees Bob's DH public value, computes K_MB."""
        raw = dh_shared(bob_public, self.priv_to_bob)
        self.key_bob = dh_to_key(raw)
        print(f'[Mallory] intercepted Bob   public={bob_public} -> K_MB={self.key_bob.hex()[:20]}...')
        # Returns Mallory's public to be forwarded to Alice (as if from Bob)
        return self.pub_to_alice

    def relay_alice_to_bob(self, pkt: dict) -> dict:
        """Decrypt from Alice, re-encrypt for Bob."""
        tag = bytes.fromhex(pkt['tag'])
        if verify_tag(self.key_alice, pkt['message'].encode(), tag):
            print(f'[Mallory] READ from Alice: "{pkt["message"]}"')
            # Re-sign for Bob
            new_tag = make_tag(self.key_bob, pkt['message'].encode())
            return {'from': 'Alice', 'message': pkt['message'], 'tag': new_tag.hex()}
        return pkt

    def relay_bob_to_alice(self, pkt: dict) -> dict:
        """Decrypt from Bob, re-encrypt for Alice."""
        tag = bytes.fromhex(pkt['tag'])
        if verify_tag(self.key_bob, pkt['message'].encode(), tag):
            print(f'[Mallory] READ from Bob: "{pkt["message"]}"')
            new_tag = make_tag(self.key_alice, pkt['message'].encode())
            return {'from': 'Bob', 'message': pkt['message'], 'tag': new_tag.hex()}
        return pkt


print('=== Participants ===')
alice5   = DHParty('Alice')
bob5     = DHParty('Bob')
mallory5 = MitMallory()

=== Participants ===
[Alice] private=7, public=17
[Bob] private=11, public=22
[Mallory] priv_a=20 pub_a=12
[Mallory] priv_b=8  pub_b=16


In [24]:
# Run the full MitM attack

print('\n=== DH Handshake (Mallory intercepts) ===')

# Step 1: Alice sends her public value - Mallory intercepts
print('\n-- Step 1: Alice broadcasts public value --')
fwd_to_bob = mallory5.intercept_alice_public(alice5.public)

# Step 2: Bob sends his public value - Mallory intercepts
print('\n-- Step 2: Bob broadcasts public value --')
fwd_to_alice = mallory5.intercept_bob_public(bob5.public)

# Step 3: Alice receives what she thinks is Bob's public (actually Mallory's)
print('\n-- Step 3: Alice finishes handshake with Mallory-as-Bob --')
alice5.finish_handshake(fwd_to_alice, claimed_from='Bob')

# Step 4: Bob receives what he thinks is Alice's public (actually Mallory's)
print('\n-- Step 4: Bob finishes handshake with Mallory-as-Alice --')
bob5.finish_handshake(fwd_to_bob, claimed_from='Alice')

print()
print('=== Key summary ===')
print(f'Alice session key  : {alice5.session_key.hex()[:24]}...')
print(f'Bob   session key  : {bob5.session_key.hex()[:24]}...')
print(f'Mallory key_alice  : {mallory5.key_alice.hex()[:24]}...')
print(f'Mallory key_bob    : {mallory5.key_bob.hex()[:24]}...')
print()
print('Alice and Bob have DIFFERENT session keys.')
print('Mallory holds BOTH.')


=== DH Handshake (Mallory intercepts) ===

-- Step 1: Alice broadcasts public value --
[Mallory] intercepted Alice public=17 -> K_MA=a3ecde0c1d9daa6b7a94...

-- Step 2: Bob broadcasts public value --
[Mallory] intercepted Bob   public=22 -> K_MB=ec4916dd28fc4c10d78e...

-- Step 3: Alice finishes handshake with Mallory-as-Bob --
[Alice] received public=12 (from "Bob")
[Alice] session key = a3ecde0c1d9daa6b7a949c87...

-- Step 4: Bob finishes handshake with Mallory-as-Alice --
[Bob] received public=16 (from "Alice")
[Bob] session key = ec4916dd28fc4c10d78e287c...

=== Key summary ===
Alice session key  : a3ecde0c1d9daa6b7a949c87...
Bob   session key  : ec4916dd28fc4c10d78e287c...
Mallory key_alice  : a3ecde0c1d9daa6b7a949c87...
Mallory key_bob    : ec4916dd28fc4c10d78e287c...

Alice and Bob have DIFFERENT session keys.
Mallory holds BOTH.


In [25]:
# Show Mallory reading and modifying the conversation

print('=== Alice sends a message to "Bob" ===')
pkt_a = alice5.send('Meet at the north gate at noon')
print(f'[Alice] sent with key {alice5.session_key.hex()[:16]}...')

# Mallory modifies the message before forwarding
tampered_msg = 'Meet at the south gate at midnight'
tampered_tag = make_tag(mallory5.key_bob, tampered_msg.encode())
tampered_pkt = {'from': 'Alice', 'message': tampered_msg, 'tag': tampered_tag.hex()}

print(f'[Mallory] READ original: "{pkt_a["message"]}"')
print(f'[Mallory] INJECTING:    "{tampered_msg}"')

bob5.receive(tampered_pkt)

print()
print('Bob accepted a message he believes is from Alice.')
print('Alice never sent that message.')
print('Neither Alice nor Bob detected anything wrong.')

=== Alice sends a message to "Bob" ===
[Alice] sent with key a3ecde0c1d9daa6b...
[Mallory] READ original: "Meet at the north gate at noon"
[Mallory] INJECTING:    "Meet at the south gate at midnight"
[Bob] ACCEPTED "Meet at the south gate at midnight" (believes from Alice)

Bob accepted a message he believes is from Alice.
Alice never sent that message.
Neither Alice nor Bob detected anything wrong.


### Why DH Alone Cannot Stop This

DH provides **computational security** against a passive eavesdropper who sees `A` and `B` as such an eavesdropper cannot compute `g^(ab) mod p` (as per the Discrete Logarithm Problem).

But DH does not provide **authentication**(one of you also asked this question during our first meet):

```
Alice does not bind A = g^a to her identity.
Bob does not bind B = g^b to his identity.
Mallory can substitute her own values freely as a result
```

| What DH provides | What DH does NOT provide |
|---|---|
| Shared secret against eavesdroppers | Proof of who you're talking to |
| Forward secrecy (ephemeral DH) | Authentication of public values |
| Computational hardness | Any protection against active attackers |

---
### Fixes

**1. Certificates (PKI)**  
A trusted Certificate Authority signs `(Alice, A)` and `(Bob, B)`.  
Anyone can verify the signature with the CA's public key.  
Mallory cannot substitute her public value without breaking the CA's signature.

**2. Authenticated DH (STS, SIGMA)**  
After the DH exchange, each party signs the transcript and verifies the other's signature.  
This is what TLS 1.3, Signal, and WireGuard do.

**3. Key Confirmation**  
Even without signatures, both parties can prove they derived the *same* key:
```
Alice --> Bob:  HMAC(K, 'alice_confirms' || transcript)
Bob   --> Alice: HMAC(K, 'bob_confirms'   || transcript)
```
This detects MitM because Mallory holds different keys and cannot produce
the correct confirmation tags for both sides simultaneously.

In [26]:
# Key confirmation -- detecting MitM without certificates

def key_confirmation(key: bytes, role: str, transcript: bytes) -> bytes:
    """Produce a key-confirmation tag."""
    label = f'{role}_confirms'.encode()
    return make_tag(key, label + transcript)


# Simulate transcript = concatenation of both public values
# (In a real protocol this would be a hash of the full handshake.)
transcript_alice = (fwd_to_alice).to_bytes(4,'big') + alice5.public.to_bytes(4,'big')
transcript_bob   = bob5.public.to_bytes(4,'big')    + (fwd_to_bob).to_bytes(4,'big')

# Alice sends confirmation to Bob
conf_alice = key_confirmation(alice5.session_key, 'alice', transcript_alice)
print(f'Alice confirmation tag: {conf_alice.hex()[:32]}...')

# Bob tries to verify it (Bob has a DIFFERENT session key!)
expected = key_confirmation(bob5.session_key, 'alice', transcript_bob)
match = hmac.compare_digest(conf_alice, expected)
print(f'Bob verifies Alice confirmation: {match}')
print()
if not match:
    print('HANDSHAKE ABORTED: key mismatch detected.')
    print('Alice and Bob now know a MitM is present.')
    print('Mallory cannot fix this as she holds different keys on each side.')

Alice confirmation tag: 7141b0793d713bc87d5c4c11c30eac0a...
Bob verifies Alice confirmation: False

HANDSHAKE ABORTED: key mismatch detected.
Alice and Bob now know a MitM is present.
Mallory cannot fix this as she holds different keys on each side.


---
<a id='sec4'></a>
## Summary

| Attack | Root cause | Minimal fix | What it leads to |
|---|---|---|---|
| **Replay** | No freshness in HMAC | Counters / nonces | Replay windows (IPsec), session IDs (TLS) |
| **Key Compromise** | Global key, no containment | Pairwise keys | Key predistribution |
| **MitM on DH** | DH has no authentication | Key confirmation | Certificates, authenticated DH, TLS 1.3 |

Each such protocol feature exists because smart people exploited vulnerabilities through the attacks we just simulated.